# Network Intrusion Detection — Progressive Dataset Evaluation

**Paper:** Chua & Salam (2023), *Evaluation of ML Algorithms in Network-Based Intrusion Detection Using Progressive Dataset*, Symmetry 15, 1251

**Setup:** Run this header cell first every time you open a new Colab session.

In [ ]:
# ── Header cell: run this first in every new Colab session ──────────────────
import sys, os

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repo so src/ modules are importable
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ← update
REPO_DIR = '/content/ids-project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

print('Environment ready.')

In [ ]:
# ── Global configuration — only line you change between runs ─────────────────
DATA_DIR = '/content/drive/MyDrive/ids-data/'  # ← set to your Drive folder

SEED = 42
SUBSAMPLE_FRAC = 0.10  # 10% of each day's CSV, read at load time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'DATA_DIR = {DATA_DIR}')

---
## §1 — Data Loading & Initial Inspection

Load CIC-IDS2017 (train) and CSE-CIC-IDS2018 (progressive test), keeping ~10% via chunked reading. Inspect shape, dtypes, memory usage, column names, and temporal structure.

In [ ]:
from src.data_loading import load_cic2017, load_cic2018, align_schemas

df_train = load_cic2017(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_test  = load_cic2018(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_train, df_test = align_schemas(df_train, df_test)

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)

### 2.3 — Shape, dtypes, memory, and column name inspection

In [ ]:
# ── Shape and memory ──────────────────────────────────────────────────────────
for name, df in [('Train (CIC-IDS2017)', df_train), ('Test  (CSE-CIC-IDS2018)', df_test)]:
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name}: {df.shape[0]:>7,} rows × {df.shape[1]} cols  |  {mem_mb:.1f} MB")

print()

# ── Dtype breakdown ───────────────────────────────────────────────────────────
print("Train dtype counts:")
print(df_train.dtypes.value_counts().to_string())
print("\nTest dtype counts:")
print(df_test.dtypes.value_counts().to_string())

In [ ]:
# ── Column name analysis ──────────────────────────────────────────────────────
# Features fall into five semantic groups derived from CICFlowMeter's documentation.
feature_groups = {
    'Packet length stats':    [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
    'Packet counts / rates':  [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
    'Inter-arrival times':    [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
    'TCP flags':              [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
    'Other / port / misc':    [c for c in df_train.columns if c not in sum([
        [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
        [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
        [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
        [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
        ['Label']
    ], []) and c != 'Label'],
}

print("Feature groups (76 features total after schema alignment):\n")
for group, cols in feature_groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

print(f"\nLabel column: 'Label' — unique values in train: {df_train['Label'].unique()}")
print(f"                       — unique values in test:  {df_test['Label'].unique()}")

**Column interpretation:** All 76 features are statistical summaries of individual network flows computed by CICFlowMeter — there are no raw packet payloads. Features naturally group into:
- **Packet length statistics** (min/max/mean/std/variance of packet sizes in both directions): capture how much data is transferred per packet. Attack flows often show very uniform or very extreme sizes.
- **Packet counts and byte rates** (total packets/bytes forward and backward, bulk transfer stats, subflow counts): capture volume and asymmetry. DoS attacks typically show extremely high forward packet rates with little or no backward traffic.
- **Inter-arrival times (IAT)** (min/max/mean/std of time between consecutive packets, flow duration): capture timing patterns. Scanning and flooding attacks have characteristically small or highly regular IATs.
- **TCP flags** (counts of SYN/FIN/RST/PSH/ACK/URG/ECE, TCP window sizes): capture connection state. A high SYN count with few ACKs is a classic SYN flood signature. Initial window sizes can distinguish OS fingerprints.
- **Other** (Destination Port, active/idle time, header lengths): Destination port is one of the strongest single discriminators (port 80/443 = HTTP traffic, port 22 = SSH, unusual high ports = scanning).

**No timestamp column in features:** CICFlowMeter writes a `Timestamp` field to the raw CSVs, but we deliberately exclude it from the feature matrix. Using the timestamp as a predictor would constitute time leakage — a model could learn "2018 flows → malicious" rather than genuine traffic patterns. The temporal structure is encoded at the *dataset split level* (all 2017 = train, all 2018 = test), not at the feature level.

### 2.4 — Temporal structure and the progressive evaluation design

In [ ]:
import os, glob

# ── Per-day file breakdown for CIC-IDS2017 ────────────────────────────────────
cic2017_dir = os.path.join(DATA_DIR, 'cic2017')
cic2018_dir = os.path.join(DATA_DIR, 'cic2018')

print("CIC-IDS2017 source files (training set):")
for f in sorted(glob.glob(os.path.join(cic2017_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

print("\nCSE-CIC-IDS2018 source files (progressive test set):")
for f in sorted(glob.glob(os.path.join(cic2018_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

In [ ]:
# ── Attack type coverage in each dataset ─────────────────────────────────────
print("Attack types in TRAINING set (CIC-IDS2017):")
train_labels = df_train['Label'].value_counts()
print(train_labels.to_string())

print("\nAttack types in TEST set (CSE-CIC-IDS2018):")
test_labels = df_test['Label'].value_counts()
print(test_labels.to_string())

# Types present in 2017 but not 2018, and vice versa
train_types = set(df_train['Label'].unique())
test_types  = set(df_test['Label'].unique())
only_train  = train_types - test_types
only_test   = test_types  - train_types

print(f"\nAttack types ONLY in train (novel to test set from its perspective): {only_train or 'none'}")
print(f"Attack types ONLY in test  (unseen during training):                 {only_test  or 'none'}")

**Temporal structure — what time means in this project:**

The progressive evaluation design operates at two levels:

**1. Within each dataset (daily granularity):**
CIC-IDS2017 was collected over five working days (Monday 3 July – Friday 7 July 2017). Each day had a different attack scenario — Monday is benign-only, with increasingly complex attacks injected Tuesday through Friday. CSE-CIC-IDS2018 was collected over ten working days in February–March 2018 with an updated network topology. Within each dataset, the per-day file structure preserves this ordering, but since we concatenate all files for training and all files for testing, the within-dataset day ordering is not used by the models.

**2. Between datasets (the progressive gap — ~8 months):**
This is the core of the paper. All training data (2017) precedes all test data (2018) by approximately eight months. There is **zero temporal overlap** between train and test. This is not a random train/test split — it is a deliberate temporal split that simulates real deployment: a model trained on historical traffic is evaluated on future traffic it has never seen.

**Why this is harder than standard evaluation:**
- Network infrastructure changed between 2017 and 2018 (different machines, different topology).
- The distribution of benign traffic shifted (software updates, different user behaviour).
- Some attack types present in 2017 may be absent in 2018, and 2018 may contain attack variants not in the training set.

**Leakage check:** There is no risk of temporal leakage in our setup. The timestamp column from the raw CSVs is excluded from features. The model sees only flow statistics — it cannot infer which year a flow came from based on the feature values alone. The train/test split is fixed at the dataset level, with no 2018 samples in training.

### 2.5 — Data hygiene scan

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
feat_cols = [c for c in df_train.columns if c != 'Label']

train_dups = df_train.duplicated(subset=feat_cols).sum()
test_dups  = df_test.duplicated(subset=feat_cols).sum()
print(f"Duplicate feature rows — train: {train_dups:,}  |  test: {test_dups:,}")

# ── Remaining NaN / inf (should be zero after _clean) ────────────────────────
train_nan = df_train[feat_cols].isnull().sum().sum()
test_nan  = df_test[feat_cols].isnull().sum().sum()
print(f"Remaining NaN values  — train: {train_nan}  |  test: {test_nan}")

train_inf = np.isinf(df_train[feat_cols].values).sum()
test_inf  = np.isinf(df_test[feat_cols].values).sum()
print(f"Remaining inf values  — train: {train_inf}  |  test: {test_inf}")

In [ ]:
# ── Constant / near-constant features (single unique value = useless) ─────────
constant_train = [c for c in feat_cols if df_train[c].nunique() <= 1]
constant_test  = [c for c in feat_cols if df_test[c].nunique()  <= 1]
print(f"Constant features in train: {constant_train or 'none'}")
print(f"Constant features in test:  {constant_test  or 'none'}")

# Near-constant: >99.9% of values are the same
near_const_train = [c for c in feat_cols
                    if df_train[c].value_counts(normalize=True).iloc[0] > 0.999]
print(f"\nNear-constant features in train (>99.9% one value): {near_const_train or 'none'}")

In [ ]:
# ── Drop duplicates and constant features; log decisions ─────────────────────
cols_to_drop = list(set(constant_train + constant_test))

df_train_clean = df_train.drop_duplicates(subset=feat_cols).reset_index(drop=True)
df_test_clean  = df_test.drop_duplicates(subset=feat_cols).reset_index(drop=True)

if cols_to_drop:
    df_train_clean = df_train_clean.drop(columns=cols_to_drop)
    df_test_clean  = df_test_clean.drop(columns=cols_to_drop)

print(f"After deduplication:")
print(f"  Train: {len(df_train):,} → {len(df_train_clean):,} rows  "
      f"(removed {len(df_train) - len(df_train_clean):,} duplicates)")
print(f"  Test:  {len(df_test):,}  → {len(df_test_clean):,}  rows  "
      f"(removed {len(df_test) - len(df_test_clean):,} duplicates)")
if cols_to_drop:
    print(f"\nDropped constant columns: {cols_to_drop}")
else:
    print("\nNo constant columns dropped.")

In [ ]:
# ── Save cleaned frames to Drive ──────────────────────────────────────────────
from joblib import dump

dump(df_train_clean, os.path.join(DATA_DIR, 'train_clean.joblib'))
dump(df_test_clean,  os.path.join(DATA_DIR, 'test_clean.joblib'))
print("Saved train_clean.joblib and test_clean.joblib to Drive.")

---
## §2 — Exploratory Data Analysis

Class distribution, feature distributions, missing values, outliers, temporal patterns, correlation analysis.

---
## §3 — Feature Engineering

Reproduce the authors' pipeline: cleaning → balancing → binary relabeling → encoding → scaling → feature creation → feature selection.

---
## §4 — Model Training

Train DT, RF, SVM, NB, ANN, DNN with GridSearchCV (k=5). Save best models to Drive.

---
## §5 — Evaluation & Reproduction Check

In-distribution evaluation (reproduce Tables 4–6). Progressive evaluation on CSE-CIC-IDS2018 (reproduce Table 7). Side-by-side comparison with paper's numbers.

---
## §6 — Error Analysis

Misclassified examples (FPs and FNs) on the progressive test set. Patterns in errors. Cybersecurity implications.

---
## §7 — Executive Summary

*(Filled after all analysis is complete.)*

---
## §8 — Summing It Up

*(Filled after all analysis is complete.)*